# Offroad Segmentation Hackathon - Master Pipeline
This notebook runs the complete 3-phase training strategy to reach 65-75% mIoU on Google Colab.

In [ ]:
# 1. Setup & Mount Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install -q segmentation-models-pytorch albumentations opencv-python-headless
!pip install -q ttach tensorboard matplotlib seaborn scikit-learn PyYAML tqdm

import os
import time
import csv
import random
from pathlib import Path
from typing import Optional, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

### 2. Dataset Preparation (Folder Copy)
This cell copies the folders from your Drive to the local Colab disk to ensure high-speed training.

In [ ]:
import os

# Paths from your Google Drive upload
# Based on your screenshot, the structure looks like this nested folder
DRIVE_TRAIN_ROOT = '/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset'
DRIVE_TEST_ROOT  = '/content/drive/MyDrive/Offroad_Segmentation_testImages/Offroad_Segmentation_testImages'

LOCAL_DATA = '/content/data'

# CRITICAL STEP: Copying to /content/ is ~10x-20x faster than reading from Drive directly
if not os.path.exists(LOCAL_DATA):
    print("🚀 Starting copy from Drive to local disk (this takes a few minutes)...")
    os.makedirs(LOCAL_DATA, exist_ok=True)
    !cp -r "$DRIVE_TRAIN_ROOT" "$LOCAL_DATA"
    !cp -r "$DRIVE_TEST_ROOT" "$LOCAL_DATA/test"
    print("✅ Copy complete!")

# Verification and Final Paths
# We check if the folders exist with a quick test
if os.path.exists(f'{LOCAL_DATA}/train'):
    TRAIN_IMG  = f'{LOCAL_DATA}/train/Color_Images'
    TRAIN_MASK = f'{LOCAL_DATA}/train/Segmentation'
    VAL_IMG    = f'{LOCAL_DATA}/val/Color_Images'
    VAL_MASK   = f'{LOCAL_DATA}/val/Segmentation'
    TEST_IMG   = f'{LOCAL_DATA}/test/Color_Images'
else:
    # Fallback in case folder nesting is different
    TRAIN_IMG  = f'{LOCAL_DATA}/Offroad_Segmentation_Training_Dataset/train/Color_Images'
    TRAIN_MASK = f'{LOCAL_DATA}/Offroad_Segmentation_Training_Dataset/train/Segmentation'
    VAL_IMG    = f'{LOCAL_DATA}/Offroad_Segmentation_Training_Dataset/val/Color_Images'
    VAL_MASK   = f'{LOCAL_DATA}/Offroad_Segmentation_Training_Dataset/val/Segmentation'
    TEST_IMG   = f'{LOCAL_DATA}/test/Color_Images'

print(f"Data Paths Locked:\n Train: {TRAIN_IMG}\n Val: {VAL_IMG}\n Test: {TEST_IMG}")

### 3. Core Classes (Dataset, Metrics, Model, Loss)

In [ ]:
# --- Dataset Constants ---
CLASS_MAP = {100:0, 200:1, 300:2, 500:3, 550:4, 600:5, 700:6, 800:7, 7100:8, 10000:9}
CLASS_NAMES = ['Trees', 'Lush_Bushes', 'Dry_Grass', 'Dry_Bushes', 'Ground_Clutter', 'Flowers', 'Logs', 'Rocks', 'Landscape', 'Sky']
IGNORE_INDEX = 255

def remap_mask(mask_array):
    out = np.full(mask_array.shape, IGNORE_INDEX, dtype=np.uint8)
    for raw_id, cls_idx in CLASS_MAP.items():
        out[mask_array == raw_id] = cls_idx
    return out

class OffRoadDataset(Dataset):
    def __init__(self, images_dir, masks_dir=None, transform=None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir else None
        self.transform = transform
        self.image_files = sorted([f for f in self.images_dir.iterdir() if f.suffix.lower() in {'.png', '.jpg'}])
    def __len__(self): return len(self.image_files)
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = np.array(Image.open(img_path).convert('RGB'))
        if self.masks_dir:
            mask_raw = np.array(Image.open(self.masks_dir / img_path.name))
            if mask_raw.ndim == 3: mask_raw = mask_raw[:,:,0]
            mask = remap_mask(mask_raw)
            if self.transform:
                aug = self.transform(image=image, mask=mask)
                image, mask = aug['image'], aug['mask'].long()
            return image, mask
        if self.transform: image = self.transform(image=image)['image']
        return image, img_path.name

# --- Metrics ---
class SegmentationMetrics:
    def __init__(self, num_classes=10, ignore_index=255):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.reset()
    def reset(self):
        self.conf_matrix = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)
    def update(self, preds, targets):
        preds, targets = preds.cpu().numpy().flatten(), targets.cpu().numpy().flatten()
        valid = targets != self.ignore_index
        self.conf_matrix += np.bincount(targets[valid]*self.num_classes + preds[valid], minlength=self.num_classes**2).reshape(self.num_classes, self.num_classes)
    def compute(self):
        tp = np.diag(self.conf_matrix).astype(np.float64)
        fp = self.conf_matrix.sum(axis=0) - tp
        fn = self.conf_matrix.sum(axis=1) - tp
        iou = tp / (tp + fp + fn + 1e-10)
        return {'mIoU': iou[self.conf_matrix.sum(axis=1)>0].mean(), 'iou_per_class': iou}

# --- Loss Functions ---
class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        valid = (targets != IGNORE_INDEX)
        target_oh = F.one_hot(targets.clamp(0, 9), 10).permute(0,3,1,2).float() * valid.unsqueeze(1)
        prob = probs * valid.unsqueeze(1)
        inter = (prob * target_oh).sum(dim=(0,2,3))
        card = prob.sum(dim=(0,2,3)) + target_oh.sum(dim=(0,2,3))
        return (1.0 - (2.0 * inter + 1.0) / (card + 1.0)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, weight=None): 
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight, ignore_index=IGNORE_INDEX)
        self.dice = DiceLoss()
    def forward(self, logits, targets, dice_w=0.5):
        return (1-dice_w)*self.ce(logits, targets) + dice_w*self.dice(logits, targets)

# --- Utils ---
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class CSVLogger:
    def __init__(self, path):
        self.path = path
        Path(path).parent.mkdir(parents=True, exist_ok=True)
    def log(self, row):
        write_header = not os.path.exists(self.path)
        with open(self.path, 'a') as f: 
            w = csv.DictWriter(f, fieldnames=row.keys())
            if write_header: w.writeheader()
            w.writerow(row)

### 4. Upgrade Config & Functions

In [ ]:
import segmentation_models_pytorch as smp

SAVE_DIR = '/content/drive/MyDrive/hackathon_runs'
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

def get_config():
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    if gpu_mem > 30: # A100
        return {'size': (768, 768), 'batch': 16, 'backbone': 'resnet101', 'workers': 2}
    elif gpu_mem > 12: # T4
        return {'size': (640, 640), 'batch': 8, 'backbone': 'resnet101', 'workers': 2}
    return {'size': (512, 512), 'batch': 4, 'backbone': 'resnet50', 'workers': 2}

CFG = get_config()
print(f"Config selected: {CFG}")

def get_transforms(size, is_train=True):
    if not is_train: 
        return A.Compose([A.Resize(size[0], size[1]), A.Normalize(), ToTensorV2()])
    return A.Compose([
        A.Resize(size[0], size[1]),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2),
        A.RandomBrightnessContrast(p=0.5),
        A.ColorJitter(p=0.4), A.GaussianBlur(p=0.2),
        A.CoarseDropout(max_holes=8, max_height=size[0]//16, max_width=size[1]//16, p=0.3),
        A.Normalize(), ToTensorV2()
    ])

def train_epoch(model, loader, criterion, optimizer, scaler, device, accum=1, dice_w=0.5):
    model.train(); total_loss = 0
    optimizer.zero_grad()
    for i, (imgs, masks) in enumerate(tqdm(loader, leave=False)):
        imgs, masks = imgs.to(device), masks.to(device)
        with autocast('cuda'):
            loss = criterion(model(imgs), masks, dice_w=dice_w) / accum
        scaler.scale(loss).backward()
        if (i+1) % accum == 0:
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        total_loss += loss.item() * accum
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); metrics = SegmentationMetrics(10); total_loss = 0
    for imgs, masks in tqdm(loader, leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        total_loss += criterion(logits, masks).item()
        metrics.update(logits.argmax(1), masks)
    return total_loss / len(loader), metrics.compute()

### 5. EXECUTION: Phase 1, 2 & 3

In [ ]:
set_seed(42)
device = torch.device('cuda')
model = smp.DeepLabV3Plus(encoder_name=CFG['backbone'], encoder_weights='imagenet', classes=10).to(device)
scaler = GradScaler('cuda')
logger = CSVLogger(f"{SAVE_DIR}/training_log.csv")

# Pre-compute class weights (Inverse frequency)
print("Computing class weights...")
temp_ds = OffRoadDataset(TRAIN_IMG, TRAIN_MASK, transform=None)
counts = np.zeros(10)
for i in range(0, len(temp_ds), 20): # sample for speed
    _, m = temp_ds[i]; counts += np.bincount(m.flatten(), minlength=11)[:10]
weights = torch.tensor(counts.sum() / (10 * counts + 1e-6), dtype=torch.float32).to(device)
weights = (weights / weights.mean()).clamp(min=0.1, max=8.0)
criterion = CombinedLoss(weight=weights)

# ------------------------------------------------------------
# PHASE 1: Stable Start (CE-Heavy)
# ------------------------------------------------------------
print("\n--- Starting Phase 1 (100 Epochs, CE-heavy) ---")
train_loader = DataLoader(OffRoadDataset(TRAIN_IMG, TRAIN_MASK, get_transforms(CFG['size'])), batch_size=CFG['batch'], shuffle=True, num_workers=CFG['workers'])
val_loader = DataLoader(OffRoadDataset(VAL_IMG, VAL_MASK, get_transforms(CFG['size'], False)), batch_size=CFG['batch'], num_workers=CFG['workers'])
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

best_miou = 0
for epoch in range(100):
    loss = train_epoch(model, train_loader, criterion, optimizer, scaler, device, dice_w=0.2)
    vloss, res = validate(model, val_loader, criterion, device)
    miou = res['mIoU']
    scheduler.step(miou)
    print(f"E{epoch:02d} | Loss: {loss:.4f} | Val mIoU: {miou*100:.2f}%")
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_p1.pth")
        print("  ✓ Saved Best P1")
    if optimizer.param_groups[0]['lr'] < 1e-6: break

# ------------------------------------------------------------
# PHASE 2: rare-class & Dice focus
# ------------------------------------------------------------
print("\n--- Starting Phase 2 (Load P1, Add Dice & Oversampling) ---")
model.load_state_dict(torch.load(f"{SAVE_DIR}/best_p1.pth"))
# Use Weighted Sampling for Logs(6), Rocks(7), GroundClutter(4)
s_weights = []; print("Computing sampler weights...")
for i in tqdm(range(len(temp_ds))):
    _, m = temp_ds[i]; w = 1.0
    if (m==6).sum() > 100: w=4.0
    elif (m==7).sum() > 100: w=3.0
    elif (m==4).sum() > 100: w=2.0
    s_weights.append(w)
sampler = WeightedRandomSampler(s_weights, len(s_weights))
train_loader = DataLoader(OffRoadDataset(TRAIN_IMG, TRAIN_MASK, get_transforms(CFG['size'])), batch_size=CFG['batch'], sampler=sampler, num_workers=CFG['workers'])

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
best_miou = 0
for epoch in range(60):
    loss = train_epoch(model, train_loader, criterion, optimizer, scaler, device, dice_w=0.5)
    vloss, res = validate(model, val_loader, criterion, device)
    miou = res['mIoU']
    print(f"E{epoch:02d} | Loss: {loss:.4f} | Val mIoU: {miou*100:.2f}%")
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_p2.pth")
        print("  ✓ Saved Best P2")

# ------------------------------------------------------------
# PHASE 3: High resolution push (768x768)
# ------------------------------------------------------------
print("\n--- Starting Phase 3 (Load P2, 768x768 Push) ---")
model.load_state_dict(torch.load(f"{SAVE_DIR}/best_p2.pth"))
P3_SIZE = (768, 768) if torch.cuda.get_device_properties(0).total_mem / 1e9 > 20 else (640, 640)
train_loader = DataLoader(OffRoadDataset(TRAIN_IMG, TRAIN_MASK, get_transforms(P3_SIZE)), batch_size=2, sampler=sampler, num_workers=CFG['workers'])
val_loader = DataLoader(OffRoadDataset(VAL_IMG, VAL_MASK, get_transforms(P3_SIZE, False)), batch_size=2, num_workers=CFG['workers'])

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
best_miou = 0
for epoch in range(30):
    loss = train_epoch(model, train_loader, criterion, optimizer, scaler, device, accum=4, dice_w=0.5)
    vloss, res = validate(model, val_loader, criterion, device)
    miou = res['mIoU']
    print(f"E{epoch:02d} | Loss: {loss:.4f} | Val mIoU: {miou*100:.2f}%")
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), f"{SAVE_DIR}/final_model.pth")
        print("  ✓ Saved Final Model")

### 6. Final TTA & Submission

In [ ]:
import ttach as tta
model.load_state_dict(torch.load(f"{SAVE_DIR}/final_model.pth"))
tta_model = tta.SegmentationTTAWrapper(model, tta.aliases.flip_transform(), merge_mode='mean')

test_ds = OffRoadDataset(TEST_IMG, transform=get_transforms(P3_SIZE, False))
test_loader = DataLoader(test_ds, batch_size=1)
pred_dir = '/content/predictions'
os.makedirs(pred_dir, exist_ok=True)

INV_MAP = {v:k for k,v in CLASS_MAP.items()}

print("Generating final predictions with TTA...")
for img, name in tqdm(test_loader):
    name = name[0]
    with torch.no_grad():
        out = tta_model(img.to(device)).argmax(1).cpu().numpy()[0]
    # Restore raw pixel IDs
    h, w = out.shape
    final = np.zeros_like(out, dtype=np.int32)
    for c, r in INV_MAP.items(): final[out==c] = r
    Image.fromarray(final.astype(np.uint16)).save(f"{pred_dir}/{name}")

!zip -r submission.zip /content/predictions
from google.colab import files
files.download('submission.zip')